# SINACON — xem từng bước trên MỘT tủ

Notebook này để **hiểu** dữ liệu và mô hình, không phải để chạy cả nhà máy. Mỗi ô có lời giải thích ở trên và bảng/hình ở dưới.

**Cách chạy:** bấm lần lượt từng ô từ trên xuống (nút ▶ bên trái ô). Đọc kết quả xong mới sang ô tiếp.

- Dữ liệu nhóm (Raw data all) **chỉ đọc**. Kết quả ghi vào `MyDrive/SINACON_mo_hinh_lai/tung_buoc/`.
- Colab báo lỗi `NameError` hoặc bị ngắt: chạy lại ô 1, ô 2 rồi ô đang dở.
- **Không** bấm lưu notebook vào GitHub.

In [ ]:
#@title 1. Kết nối Drive và lấy code
import subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
# Code nằm ở nhánh fault-diagnosis (repo công khai, không cần token)
REPO_URL = 'https://github.com/phanthutrang410/OPTIMA-Core.git'
BRANCH = 'fault-diagnosis'
CODE_DIR = Path('/content/OPTIMA-Core')
cmds = ([['git', '-C', str(CODE_DIR), 'fetch', 'origin', BRANCH],
         ['git', '-C', str(CODE_DIR), 'checkout', '-B', BRANCH, 'origin/' + BRANCH]]
        if (CODE_DIR / '.git').is_dir() else [['git', 'clone', '--branch', BRANCH, REPO_URL, str(CODE_DIR)]])
for c in cmds:
    subprocess.run(c, check=True)
sys.path.insert(0, str(CODE_DIR / 'src'))
import importlib, sinacon.explore
importlib.reload(sinacon.explore)
print('Code version:', subprocess.check_output(['git', '-C', str(CODE_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
#@title 2. Chọn tủ và tháng log
TU = '9.3'  #@param {type:'string'}
THANG = 'Tháng 8–10/2026 (13/08–31/10)'  #@param ['Tháng 10/2025', 'Tháng 1–3/2026 (12/01–12/03)', 'Tháng 4–6/2026 (04/04–03/06)', 'Tháng 6–8/2026 (13/06–13/08)', 'Tháng 8–10/2026 (13/08–31/10)']
# Thư mục "Raw data all" của nhóm (chỉ đọc). Đổi link nếu dữ liệu chuyển chỗ.
DATA_LINK = 'https://drive.google.com/drive/folders/1GnXM5FI8x34fyRvBLlQ8QTJqr7UZ-7K9'  #@param {type:'string'}

import pandas as pd
from IPython.display import display
from sinacon import explore as ex
pd.set_option('display.max_colwidth', 90); pd.set_option('display.max_rows', 200)

LOG_ROOT = ex.find_period('/content/drive', DATA_LINK, THANG)
CAB = ex.find_cabinet(LOG_ROOT, TU)
OUT = Path('/content/drive/MyDrive/SINACON_mo_hinh_lai/tung_buoc') / ex.PERIODS[THANG][2] / TU
OUT.mkdir(parents=True, exist_ok=True)
files = ex.list_files(CAB)
print('Thư mục log của tủ:', CAB)
print(f'{len(files)} file log ngày, tổng {files.MB.sum():.0f} MB. Kết quả sẽ ghi vào:', OUT)
display(files.head(10))

## Bước 0 — Trong một file log có gì?

Mỗi tủ ghi **mỗi ngày một (vài) file** `APS-xxxxxx_YYYYMMDD_HHMMSS.csv`. Trong file trộn nhiều **loại dòng**; chữ đầu dòng cho biết loại:
**APU** = ghi cho từng khối công suất (APU 1–4), **APS** = ghi cho cả tủ; **Trig** = chỉ ghi khi có thay đổi, **10s / 60s** = ghi đều mỗi 10 giây / mỗi phút.

Bảng ô 3 có 3 cột giải thích:
- `cot_chinh`: loại dòng đó có những cột số đo nào (lấy từ dòng tiêu đề trong log; dấu (?) = đoán theo ký hiệu, chưa có tài liệu Siemens).
- `mo_hinh_hien_tai`: mô hình chẩn đoán hiện tại có đọc không. **Hiện chỉ đọc 3 loại**: `APU Stat Trig`, `APS Stat Trig`, `APU Stat 60s`.
- `co_the_dung_cho`: đọc thêm thì giúp được gì. "nên xét" = có số đo cho thiết bị mà mô hình đang thiếu căn cứ.

Đây là **lựa chọn ban đầu, chưa chốt**; các loại "nên xét" để dành cho phần xem lại mô hình vật lý.
Mã của `APU Stat Trig` là mã **B** trong bảng phân nhóm; mã của `APS Stat Trig` là mã **A**.

In [ ]:
#@title 3. Xem các loại dòng và vài dòng mẫu (ngày đầu tiên)
rt = ex.record_types(CAB)
print('File:', rt.attrs['file']); display(rt)
print('\n--- APU Stat Trig: 5 dòng đầu (cột Error1–8 = mã lỗi đang bật, 0 = không có) ---')
x = ex.sample_lines(CAB, 'APU Stat Trig', 5); display(x.iloc[:, [1, 2, 4] + list(range(15, 23))])
print('\n--- APS Stat Trig: 5 dòng có mã lỗi/warning ---')
display(ex.sample_lines(CAB, 'APS Stat Trig', 5, only_codes=True).iloc[:, 1:])
print('\n--- APU Stat 60s: 5 dòng đầu ---')
display(ex.sample_lines(CAB, 'APU Stat 60s', 5).iloc[:, 1:7])

## Bước 0b — Trạng thái OpState của APU

Theo manual SINACON PV (01/2020, tr. 212–213). APU đi từ trên xuống khi khởi động:

| OpState | Nghĩa |
|---|---|
| 20, 30 | Bộ điều khiển APU đang khởi động lại (20 không có trong manual) |
| 50 | Tắt: chưa nối DC, chưa nối AC |
| 100–200 | Đang nối phía DC (tấm pin) |
| 300–400 | Đang nối phía AC (lưới) |
| 500 | Đã nối AC nhưng **PWM tắt: không phát**. Mã lỗi bật lúc 500 = APU vừa dừng phát |
| 3000, 3040–3060 | Đang phát điện |
| 3010 / 3020 / 3030 | Đang phát nhưng bị giới hạn công suất / dòng / áp DC |

Cả tủ (APS) ghi OpState 20000, không có trong manual.

Ô dưới đọc **cả tháng** của tủ và liệt kê **mỗi lần một mã lỗi bật**, kèm: ở APU nào, APU đang ở trạng thái gì, và mã đó thuộc
**thiết bị nào / lỗi gì** theo bảng phân nhóm của bạn.

In [ ]:
#@title 4. Mọi lần mã lỗi bật trong tháng (đọc khoảng 1–3 phút)
cabinet, errors, warnings = ex.read_cabinet(CAB)
events = ex.code_events(errors)
print(f'Tủ {TU} ({cabinet}): {len(events)} lần mã lỗi bật trong tháng.')
print('\nCác mã đã gặp, xếp theo số lần bật:')
display(ex.code_summary(events))
print('\n30 lần bật đầu tiên:')
display(events.head(30))
events.to_csv(OUT / 'ma_loi_bat.csv', index=False, encoding='utf-8-sig')

In [ ]:
#@title 5. Vẽ một ngày: trạng thái 4 APU và nhiệt độ
NGAY = ''  #@param {type:'string'}
# Để trống = ngày có nhiều mã lỗi nhất. Hoặc gõ ngày dạng 2026-10-15.
ngay = NGAY or str(events['t'].dt.date.value_counts().index[0])
print('Ngày:', ngay)
fig = ex.plot_day(CAB, errors, events, ngay)

## Bảng phân nhóm hai tầng (bảng của bạn)

**Tầng 1 = thiết bị nào**, **tầng 2 = thiết bị đó bị gì**. 167 mã error, mỗi mã đúng một ô.
Thêm lớp **Vận hành** cho cảnh báo không do hỏng thiết bị (8 warning: APU chế độ tay, bảo trì, bị tắt).
Các warning khác không dùng để đoán thiết bị.

In [ ]:
#@title 6. Bảng thiết bị (tầng 1) và loại lỗi (tầng 2)
display(ex.device_table())

## Gom mã thành sự cố

Một lần hỏng thường bật **nhiều mã liền nhau** (ví dụ mất Powerlink + nguồn 24 V + AC switch). Mô hình gom chúng thành **một sự cố**:

1. Mỗi mã tính từ lúc bật tới lúc tắt.
2. Mã bật trong vòng **G phút** sau khi mã trước tắt → cùng sự cố.
3. Thời điểm sự cố **t0** = lúc mã đầu tiên bật. *Thiết bị của mã đầu* = ô của mã đó trong bảng.
4. Warning của tủ bật trong **[t0 − B giây, t0 + A giây]** được ghi kèm.

G, B, A **không đặt tay**: tính bằng công thức trên dữ liệu đợt **xây phương pháp 04/04–03/06/2026, 16 tủ**
(1.1, 1.2, 4.1, 4.2, 7.1, 7.2, 10.1, 10.2, 13.1, 13.2, 16.1, 16.2, 19.1, 19.2, 22.1, 22.2), lưu ở `config/windows.json`
(code: `src/sinacon/calibrate.py`). Hiện tại: **G = 2 phút, B = 10 giây, A = 110 giây** (ô 6a in ra giá trị đang dùng).

### Công thức chọn G (ngưỡng gom)
- Thử G = 0; 0,5; 1; 2; 5; 10; 15; 30; 60 phút, đếm số sự cố N(G). **G càng lớn → gộp càng nhiều → N càng nhỏ.**
- Chọn **G nhỏ nhất mà tăng lên mức kế tiếp chỉ bớt thêm dưới 2 % số sự cố**: tới đó chuỗi đã gom xong; tăng nữa chỉ gộp nhầm chuyện khác.
- Đợt 04–06, 16 tủ: N = 430 / 131 / 122 / 119 / 117 / 115 ở 0 / 0,5 / 1 / 2 / 5 / 10 phút → từ 2 lên 5 phút chỉ bớt 1,7 % → **G = 2 phút**.
- Đường gần phẳng từ 1 phút trở đi, nên chọn 2 hay 5 phút gần như không đổi kết quả (119 so với 117 sự cố).

### Công thức chọn B, A (cửa sổ warning)
- Với mỗi lần warning bật, đo nó cách t0 của sự cố gần nhất bao nhiêu giây.
- **Mức nền** = số warning bật mỗi giây ở xa sự cố (5–10 phút).
- Chia trục thời gian thành ô 10 giây; từ t0 đi ra hai phía, giữ các ô liền nhau có mật độ **≥ 2 × mức nền**. Mép ngoài = B (phía trước), A (phía sau).
- Đợt 04–06: **−10 s … +110 s**. Đối chiếu (không dùng để chọn): 12/01–12/03: −10…+120; 13/06–13/08: 0…+70; 13/08–31/10: −20…+110.

**Muốn đổi cách chọn** (vd 3 × nền thay vì 2 ×): sửa `RULE` trong `calibrate.py`, chạy lại trên đợt 04–06, ghi nhật ký. Không chọn lại trên đợt kiểm định.

Ô 6a, 6b vẽ cho **tủ và đợt đang chọn ở ô 2** để chị tự xem; con số đang dùng vẫn là số chọn trên đợt 04–06.


In [ ]:
#@title 6a. Ngưỡng gom: giá trị đang dùng và thử nhiều ngưỡng trên tủ này
from sinacon import calibrate as cal
w = cal.load_windows()
print('Đang dùng:', w['incident_gap_min'], 'phút gom;', 'warning từ −', w['warn_before_s'], 's tới +', w['warn_after_s'], 's')
print('Chọn trên:', w.get('data', {}).get('period', 'giá trị đặt tay cũ'), '|', ', '.join(w.get('data', {}).get('cabinets', [])))
cabinet, errors, warnings = ex.read_cabinet(CAB)   # đọc lại nếu chưa chạy ô 4
fig, tab = ex.plot_gap_choice(errors)
display(tab)


In [ ]:
#@title 6b. Vì sao cửa sổ warning −10 s / +90 s: warning bật cách t0 bao lâu
fig, bang, trong, xa = ex.plot_warning_window(errors, warnings)
display(bang)
print('Warning hay gặp TRONG cửa sổ (đi kèm sự cố):'); display(trong)
print('Warning hay gặp CÁCH sự cố > 10 phút (chuyện riêng):'); display(xa)


In [ ]:
#@title 7. Danh sách sự cố của tủ
inc = ex.incidents_of(CAB, TU)
print(f'{len(inc)} sự cố.')
display(inc[['t0', 'apu', 'thiet_bi_ma_dau', 'first_fault_type', 'codes']])
inc.to_csv(OUT / 'su_co.csv', index=False, encoding='utf-8-sig')

## Bước A — Cảnh báo giả hay thật?

Nhìn **hậu quả** quanh t0, xét theo thứ tự, gặp quy tắc nào trước thì dừng:

1. Lúc t0 không APU nào đang phát (đêm, tủ đã dừng) → không đo được, **tạm để thật**.
2. Có APU không phát suốt từ 3 giờ trước tới 30 phút sau trong khi APU khác phát → **thật**: APU hỏng/tách kéo dài.
3. Ngoài 08:00–16:30, mọi APU về thẳng Tắt (50), không mã phần cứng → **giả**: dừng do trời tối.
4. Có APU đang phát bị ngắt quanh t0 → **thật** (mất ≤ 10 APU·phút: ngắt ngắn, tự phục hồi).
5. Mã nhiệt, nhiệt độ chạm ngưỡng bảo vệ (IGBT 107 °C, cuộn cảm ~160 °C) hoặc APU bị giới hạn công suất → **thật**.
6. Mã nhiệt, cảm biến nhảy > 15 °C/phút → **giả** (cảm biến).
7. Mã nhiệt, nhiệt còn xa ngưỡng, không ngắt → **giả**.

*Mã nhiệt* và *mã phần cứng* lấy từ bảng của bạn (ô có chữ "nhiệt" + Môi trường tủ; ô quá dòng / driver).

In [ ]:
#@title 8. Chạy Bước A cho tủ này (trích số đo quanh từng sự cố, khoảng 1–3 phút)
a = ex.step_a(inc, CAB, TU, OUT)
print(a['ket_luan'].value_counts().to_string(), '\n')
print(a['quy_tac'].value_counts().to_string())
cols = ['t0', 'apu', 'thiet_bi_ma_dau', 'ket_luan', 'quy_tac', 'ly_do']
display(a[cols].reset_index().rename(columns={'index': 'so'}))
a.to_csv(OUT / 'buoc_A.csv', index=False, encoding='utf-8-sig')
print('Đã lưu:', OUT / 'buoc_A.csv')

In [ ]:
#@title 9. Xem kỹ một sự cố: Bước A đã nhìn thấy gì
SO = 0  #@param {type:'integer'}
# Gõ số ở cột "so" của bảng ô 8.
r = a.iloc[SO]
print(f"Sự cố {SO}: {r['t0']}  APU {r['apu']}")
print('Mã:', r['codes'])
print('Kết luận:', r['ket_luan'], '-', r['quy_tac'])
print('Lý do:', r['ly_do'])
fig = ex.plot_incident(OUT, TU, r['t0'])

## Bước B — Thiết bị nào, lỗi gì?

*Đang làm lại theo bảng phân nhóm hai tầng; sẽ thêm vào đây khi xong.*